# Fisher Matrix Exploration Notebook

This notebook loads a single microlensing event and its posterior samples to experiment with Fisher matrix stuff. Fill in the paths below and run to load your data.

## 1. Set File Paths

Fill in these paths with your specific event data:

In [1]:
# ==== FILL IN THESE PATHS ====

# Path to the data directory (e.g., Fisher_overguide_m40)
data_path = "/Users/malpas.1/Code/GullsPosteriors/Fisher_overguide_m40/"

# Specific lightcurve file to load (.det.lc file)
lightcurve_file = "5f_overguide_m40_1_721_311.det.lc"  # e.g., "5f_overguide_m40_1_721_321.det.lc"

# Posterior samples file (.npy file) - leave empty if not yet generated
samples_file = "posteriors/721_1_311_post_samples.npy"  # e.g., "posteriors/721_1_321_post_samples.npy"

# Event name for plots
event_name = "721_1_311"  # e.g., "721_1_321"

# =================================

import os
print(f"Data path: {data_path}")
print(f"Lightcurve file: {lightcurve_file}")
print(f"Samples file: {samples_file}")
print(f"Event name: {event_name}")

Data path: /Users/malpas.1/Code/GullsPosteriors/Fisher_overguide_m40/
Lightcurve file: 5f_overguide_m40_1_721_311.det.lc
Samples file: posteriors/721_1_311_post_samples.npy
Event name: 721_1_311


## 2. Import Required Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import corner
import pickle
import warnings
warnings.filterwarnings('ignore')

# Import the GullsPosteriors modules
from Data import Data
from Parallax import Parallax
from Event import Event
from Fit import Fit
from Orbit import Orbit

print("Libraries imported successfully!")

Libraries imported successfully!


## Actual Data File Structure (40 columns)

Based on analysis of the actual lightcurve file, here's what we found:

### Columns 1-20: Basic Lightcurve Data
| Column | Name | Description |
|--------|------|-------------|
| 1 | Simulation_time | Simulation time |
| 2 | measured_relative_flux | Observed flux (with noise) |
| 3 | measured_relative_flux_error | Flux measurement error |
| 4 | true_relative_flux | True flux (no noise) |
| 5 | true_relative_flux_error | True flux error |
| 6 | observatory_code | Filter/observatory ID (0-4) |
| 7 | saturation_flag | Saturation indicator |
| 8 | best_single_lens_fit | Single lens fit quality |
| 9 | parallax_shift_t | Parallax shift (tangential) |
| 10 | parallax_shift_u | Parallax shift (radial) |
| 11 | BJD | Barycentric Julian Date |
| 12-17 | source_x, source_y, lens1_x, lens1_y, lens2_x, lens2_y | Astrometric positions |
| 18-20 | parallax_shift_x, parallax_shift_y, parallax_shift_z | 3D parallax shifts |

### Columns 21-30: Fisher Derivatives (Model Parameters)
Derivatives of flux w.r.t. the 9 binary lens model parameters:

| Column | Parameter | Space | Description |
|--------|-----------|-------|-------------|
| 21 | s | Logarithmic | Projected separation |
| 22 | q | Logarithmic | Mass ratio |
| 23 | rho | Logarithmic | Finite source effect |
| 24 | u0 | Linear | Impact parameter |
| 25 | alpha | Linear | Source trajectory angle |
| 26 | t0 | Linear | Time of maximum magnification |
| 27 | tE | Logarithmic | Einstein radius crossing time |
| 28 | piEE | Linear | Parallax (East component) |
| 29 | piEN | Linear | Parallax (North component) |

### Columns 31-40: Fisher Derivatives (Flux Parameters)
Derivatives of flux w.r.t. baseline and source flux for each filter:

| Column | Parameter | Description |
|--------|-----------|-------------|
| 31 | dF/dFbase0 | Derivative w.r.t. baseline flux (filter 0) |
| 32 | dF/dfs0 | Derivative w.r.t. source flux (filter 0) |
| 33 | dF/dFbase1 | Derivative w.r.t. baseline flux (filter 1) |
| 34 | dF/dfs1 | Derivative w.r.t. source flux (filter 1) |
| 35 | dF/dFbase2 | Derivative w.r.t. baseline flux (filter 2) |
| 36 | dF/dfs2 | Derivative w.r.t. source flux (filter 2) |
| 37 | dF/dFbase3 | Derivative w.r.t. baseline flux (filter 3) |
| 38 | dF/dfs3 | Derivative w.r.t. source flux (filter 3) |
| 39 | dF/dFbase4 | Derivative w.r.t. baseline flux (filter 4) |
| 40 | dF/dfs4 | Derivative w.r.t. source flux (filter 4) |

**Note:** 
- System is normalized with Fbase=1 for all filters
- dF/dFbase ≈ 1.0 (expected for normalized system)
- dF/dfs ≈ magnification A (flux sensitivity to source changes)
- Only derivatives for the filter corresponding to each data point are non-zero

## 3. Load and Inspect Data

Load the lightcurve data and convert it to a DataFrame for easy exploration:

In [10]:
# Initialize the Data object
data_obj = Data()

if lightcurve_file:
    # Load the specific lightcurve file
    full_lc_path = os.path.join(data_path, lightcurve_file)
    
    if os.path.exists(full_lc_path):
        print(f"Loading lightcurve from: {full_lc_path}")
        
        # Let's actually understand what's in this file
        print("\n" + "="*60)
        print("ANALYZING LIGHTCURVE FILE STRUCTURE")
        print("="*60)
        
        with open(full_lc_path, 'r') as f:
            lines = f.readlines()
        
        print(f"Total lines in file: {len(lines)}")
        
        # Look for header information and obs group info
        header_lines = []
        obs_group_info = []
        data_start = 0
        
        for i, line in enumerate(lines):
            if line.strip().startswith('#') or line.strip().startswith('%') or not line.strip():
                header_lines.append((i, line.strip()))
                # Look for obs group information
                if 'obs group' in line.lower() or 'filter' in line.lower():
                    obs_group_info.append(line.strip())
            else:
                # First non-comment line is probably where data starts
                if data_start == 0:
                    data_start = i
        
        if header_lines:
            print(f"\nFound {len(header_lines)} header/comment lines:")
            for line_num, content in header_lines[:15]:  # Show first 15 header lines
                print(f"  Line {line_num+1}: {content}")
            if len(header_lines) > 15:
                print(f"  ... and {len(header_lines)-15} more header lines")
        
        if obs_group_info:
            print(f"\nObs Group Information:")
            for info in obs_group_info:
                print(f"  {info}")
        
        print(f"\nData appears to start at line {data_start+1}")
        
        # Analyze the actual data structure
        print(f"\nFirst 3 data lines:")
        for i in range(data_start, min(data_start+3, len(lines))):
            parts = lines[i].strip().split()
            print(f"  Line {i+1}: {len(parts)} columns")
            print(f"    First 8: {' '.join(parts[:8])}")
            if len(parts) > 8:
                print(f"    Last 10: {' '.join(parts[-10:])}")
            if len(parts) > 18:
                print(f"    Middle mystery cols (9-{len(parts)-10}): {' '.join(parts[8:-10])}")
        
        # Try to determine column structure
        sample_line = lines[data_start].strip().split()
        n_cols = len(sample_line)
        print(f"\nDetected {n_cols} columns per data line")
        
        # Expected structure analysis
        expected_basic = 8  # BJD, flux, error, parallax, true_flux, true_error, sim_time
        expected_flux_params = 10  # Fs, Fb for 5 filters each
        mystery_cols = n_cols - expected_basic - expected_flux_params
        
        print(f"\nCOLUMN BREAKDOWN ANALYSIS:")
        print(f"  Expected basic columns (8): BJD, measured_flux, error, parallax_t, parallax_u, true_flux, true_error, sim_time")
        print(f"  Expected flux parameters (10): Fs0, Fb0, Fs1, Fb1, Fs2, Fb2, Fs3, Fb3, Fs4, Fb4")
        print(f"  Mystery columns ({mystery_cols}): THESE SHOULD BE FISHER DERIVATIVES")
        print(f"  Total columns: {n_cols}")
        
        # Fisher derivatives expectation
        n_params = 9  # s, q, rho, u0, alpha, t0, tE, piEE, piEN
        print(f"\nFISHER DERIVATIVES ANALYSIS:")
        print(f"  Expected parameters: {n_params} (s, q, rho, u0, alpha, t0, tE, piEE, piEN)")
        print(f"  Mystery columns found: {mystery_cols}")
        print(f"  Expected derivatives per obs group: {n_params}")
        
        if mystery_cols == n_params:
            print(f"  ✓ PERFECT MATCH: Looks like derivatives for 1 obs group only")
        elif mystery_cols == n_params * 3:
            print(f"  ✓ POSSIBLE MATCH: Looks like derivatives for 3 obs groups")
        else:
            print(f"  ✗ MISMATCH: {mystery_cols} mystery columns doesn't match expected Fisher derivatives")
            print(f"     Possible explanations:")
            print(f"     - Derivatives for {mystery_cols/n_params:.1f} obs groups")
            print(f"     - Different parameter set")
            print(f"     - Additional metadata columns")
        
        # Show what each column likely contains
        print(f"\nLIKELY COLUMN ASSIGNMENTS:")
        col_assignments = []
        
        # Basic columns (0-7)
        basic_names = ["BJD", "measured_flux", "flux_error", "parallax_t", "parallax_u", "true_flux", "true_error", "sim_time"]
        for i, name in enumerate(basic_names):
            col_assignments.append(f"  Col {i+1:2d}: {name}")
        
        # Fisher derivatives (8 to 8+mystery_cols-1)
        param_names = ["s", "q", "rho", "u0", "alpha", "t0", "tE", "piEE", "piEN"]
        for i in range(mystery_cols):
            param_idx = i % len(param_names)
            obs_group = i // len(param_names)
            col_num = 8 + i + 1
            col_assignments.append(f"  Col {col_num:2d}: d/d{param_names[param_idx]} (obs_group_{obs_group})")
        
        # Flux parameters (last 10)
        flux_names = ["Fs0", "Fb0", "Fs1", "Fb1", "Fs2", "Fb2", "Fs3", "Fb3", "Fs4", "Fb4"]
        for i, name in enumerate(flux_names):
            col_num = n_cols - len(flux_names) + i + 1
            col_assignments.append(f"  Col {col_num:2d}: {name}")
        
        for assignment in col_assignments:
            print(assignment)
        
    else:
        print(f"Lightcurve file not found: {full_lc_path}")
        print("Available files in directory:")
        for f in os.listdir(data_path):
            if f.endswith('.det.lc'):
                print(f"  {f}")
else:
    print("Please specify a lightcurve file in the first cell")

No SIMULATION_ZERO_TIME loaded from .prm file.
Loading lightcurve from: /Users/malpas.1/Code/GullsPosteriors/Fisher_overguide_m40/5f_overguide_m40_1_721_311.det.lc

ANALYZING LIGHTCURVE FILE STRUCTURE
Total lines in file: 54452

Found 13 header/comment lines:
  Line 1: #fs: 0.0166966 0.00778817 0.0163365 0.00585184 0.00617469
  Line 2: #Sourcemag: 28.3857 26.0216 25.2154 24.6972 24.6938 24.3673 24.3754 24.3992 22.5647 23.8755 22.9231 34.1123 32.1339 29.5429 26.0978 27.9293 28.2446 26.5465 25.7557 33.7724 30.9834 28.2855 26.7233 25.4417 27.944 29.8987 26.3474 25.4439 24.7277 23.8013 22.9313 22.5833
  Line 3: #Sourcedata: 20662 -6.87566 -4.02094 109.63 118.997 -16.1005 -150.234 0.182405 0 10 3417.36 5.09398 0 0.182403 10.5234 0.199808 -0.225886 0.607363 -2.20129 268.927 -29.5414 8.06329 -0.12177 0.0854101 -0.309461 0.169711 0
  Line 4: #Obssrcmag: 24.6938 28.3857 26.0216 24.3754 24.3992
  Line 5: #Lensmag: 28.5992 26.2111 25.3999 24.8718 24.8664 24.5305 24.5411 24.5562 22.7217 24.0509 23

In [11]:
print("\n" + "="*60)
print("LOADING DATA WITH Data.load_data() METHOD")
print("="*60)

# Load the data using the Data class method
data_dict = data_obj.load_data(full_lc_path)

print(f"Successfully loaded data for {len(data_dict)} observatories: {list(data_dict.keys())}")

# Analyze what we actually got
for obs_code, obs_data in data_dict.items():
    print(f"\nObservatory {obs_code}:")
    print(f"  Data shape: {obs_data.shape} (rows=parameters, cols=time_points)")
    print(f"  Number of time points: {obs_data.shape[1]}")
    
    # Show what each row contains
    row_names = [
        "BJD (time)",
        "measured_relative_flux", 
        "measured_relative_flux_error",
        "parallax_shift_t",
        "parallax_shift_u", 
        "true_relative_flux",
        "true_relative_flux_error",
        "simulation_time"
    ]
    
    print(f"  Row breakdown:")
    for i, name in enumerate(row_names):
        if i < obs_data.shape[0]:
            values = obs_data[i, :]
            print(f"    Row {i}: {name}")
            print(f"           Range: {np.min(values):.6f} to {np.max(values):.6f}")
            print(f"           Sample: {values[:3]} ...")
        else:
            print(f"    Row {i}: {name} - MISSING")

# Show the Fisher matrix information if available
print(f"\n" + "="*60)
print("FISHER MATRIX INFORMATION")
print("="*60)

if hasattr(data_obj, 'model_derivatives') and data_obj.model_derivatives is not None:
    print(f"✓ Fisher derivatives shape: {data_obj.model_derivatives.shape}")
    print(f"✓ Fisher covariance available: {hasattr(data_obj, 'model_covariance')}")
    if hasattr(data_obj, 'model_parameter_uncertainties'):
        print(f"✓ Parameter uncertainties available: {len(data_obj.model_parameter_uncertainties)} parameters")
        print(f"  Uncertainties: {data_obj.model_parameter_uncertainties}")
else:
    print("✗ No Fisher derivative information found in this file")
    print("  This means either:")
    print("  1. The lightcurve was generated without Fisher matrix computation")
    print("  2. The file format doesn't include Fisher derivatives")
    print("  3. There was an error loading the Fisher information")

# Let's also check what the relative flux actually means
print(f"\n" + "="*60)
print("RELATIVE FLUX ANALYSIS")
print("="*60)

for obs_code, obs_data in data_dict.items():
    measured_flux = obs_data[1, :]  # measured_relative_flux
    true_flux = obs_data[5, :]      # true_relative_flux
    
    print(f"\nObservatory {obs_code}:")
    print(f"  Measured relative flux:")
    print(f"    Min: {np.min(measured_flux):.6f}, Max: {np.max(measured_flux):.6f}")
    print(f"    Mean: {np.mean(measured_flux):.6f}, Std: {np.std(measured_flux):.6f}")
    
    print(f"  True relative flux:")
    print(f"    Min: {np.min(true_flux):.6f}, Max: {np.max(true_flux):.6f}")
    print(f"    Mean: {np.mean(true_flux):.6f}, Std: {np.std(true_flux):.6f}")
    
    # Check if baseline is around 1.0 (relative flux system)
    baseline_measured = np.min(measured_flux)
    baseline_true = np.min(true_flux)
    print(f"  Baseline flux (minimum): measured={baseline_measured:.6f}, true={baseline_true:.6f}")
    
    if np.abs(baseline_measured - 1.0) < 0.1:
        print(f"  → This looks like a relative flux system (baseline ≈ 1.0)")
    else:
        print(f"  → This might be absolute flux or different normalization")

print(f"\nNow you can see exactly what data structure you're working with!")


LOADING DATA WITH Data.load_data() METHOD
Detected 80 columns in /Users/malpas.1/Code/GullsPosteriors/Fisher_overguide_m40/5f_overguide_m40_1_721_311.det.lc
Header: ['Simulation_time', 'measured_relative_flux', 'measured_relative_flux_error', 'true_relative_flux', 'true_relative_flux_error', 'observatory_code', 'saturation_flag', 'best_single_lens_fit', 'parallax_shift_t', 'parallax_shift_u', 'BJD', 'source_x', 'source_y', 'lens1_x', 'lens1_y', 'lens2_x', 'lens2_y', 'X', 'Y', 'Z', 'dTheta1', 'dTheta2', 'dTheta3', 'dTheta4', 'dTheta5', 'dTheta6', 'dTheta7', 'dTheta8', 'dTheta9', 'dTheta10', 'dTheta11', 'dTheta12', 'dTheta13', 'dTheta14', 'dTheta15', 'dTheta16', 'dTheta17']
Data columns: Index(['Simulation_time', 'measured_relative_flux',
       'measured_relative_flux_error', 'true_relative_flux',
       'true_relative_flux_error', 'observatory_code', 'saturation_flag',
       'best_single_lens_fit', 'parallax_shift_t', 'parallax_shift_u', 'BJD',
       'source_x', 'source_y', 'lens1_x

### Convert to DataFrame for Easy Exploration

In [ ]:
if 'data_dict' in locals():
    # Column names based on the load_data method documentation
    column_names = [
        'BJD',
        'measured_relative_flux', 
        'measured_relative_flux_error',
        'parallax_shift_t',
        'parallax_shift_u', 
        'true_relative_flux',
        'true_relative_flux_error',
        'Simulation_time'
    ]
    
    # Create DataFrames for each observatory
    dfs = {}
    for obs_code, obs_data in data_dict.items():
        # obs_data is shape (8, n_points), so transpose to get (n_points, 8)
        df = pd.DataFrame(obs_data.T, columns=column_names)
        df['observatory_code'] = obs_code
        dfs[obs_code] = df
        
        print(f"Observatory {obs_code}: {len(df)} data points")
        print(f"  Time range: {df['BJD'].min():.1f} to {df['BJD'].max():.1f}")
        print(f"  Flux range: {df['measured_relative_flux'].min():.3f} to {df['measured_relative_flux'].max():.3f}")
        print()
    
    # Combine all observatories into one DataFrame
    combined_df = pd.concat(dfs.values(), ignore_index=True)
    print(f"Combined DataFrame: {len(combined_df)} total data points")
    
    # Show the first few rows
    print("\nFirst 5 rows:")
    display(combined_df.head())
else:
    print("No data loaded yet. Please load lightcurve data first.")

## 4. Data Processing and Analysis

Load posterior samples if available and analyze the parameter distributions:

In [ ]:
# Load posterior samples if available
samples = None
truths = None

if samples_file and os.path.exists(os.path.join(data_path, samples_file)):
    samples_path = os.path.join(data_path, samples_file)
    print(f"Loading samples from: {samples_path}")
    samples = np.load(samples_path)
    print(f"Samples shape: {samples.shape} (n_samples, n_parameters)")
    
    # Try to load corresponding truths file
    truths_file = samples_file.replace('_post_samples.npy', 'end_truths.pkl')
    truths_path = os.path.join(data_path, truths_file)
    
    if os.path.exists(truths_path):
        print(f"Loading truths from: {truths_path}")
        with open(truths_path, 'rb') as f:
            truths = pickle.load(f)
        print(f"Truths loaded: {len(truths)} entries")
    else:
        print(f"Truths file not found: {truths_path}")
        
elif samples_file:
    print(f"Samples file not found: {os.path.join(data_path, samples_file)}")
else:
    print("No samples file specified. This is fine if you haven't run the sampler yet.")
    print("You can still explore the lightcurve data and Fisher information.")

### Fisher Matrix Analysis

In [ ]:
# Analyze Fisher matrix if available
if hasattr(data_obj, 'model_covariance') and data_obj.model_covariance is not None:
    fisher_cov = data_obj.model_covariance
    fisher_unc = data_obj.model_parameter_uncertainties
    
    print(f"Fisher covariance matrix shape: {fisher_cov.shape}")
    print(f"Fisher uncertainties: {fisher_unc}")
    
    # Parameter names (in the order used by the code)
    param_names = ['s', 'q', 'rho', 'u0', 'alpha', 't0', 'tE', 'piEE', 'piEN']
    
    # Create DataFrame for Fisher covariance
    fisher_df = pd.DataFrame(fisher_cov, index=param_names, columns=param_names)
    print("\nFisher covariance matrix:")
    display(fisher_df)
    
    # Calculate correlation matrix
    fisher_corr = fisher_cov / np.outer(fisher_unc, fisher_unc)
    corr_df = pd.DataFrame(fisher_corr, index=param_names, columns=param_names)
    print("\nFisher correlation matrix:")
    display(corr_df)
    
else:
    print("No Fisher matrix information available.")
    print("This might be because:")
    print("1. The lightcurve file doesn't contain Fisher derivative columns")
    print("2. The Fisher computation failed")
    print("3. You haven't loaded a lightcurve file yet")

## 5. Visualization

Create plots to visualize the data and posterior samples:

### Plot Lightcurve Data

In [ ]:
if 'combined_df' in locals():
    plt.figure(figsize=(12, 8))
    
    # Colors for different observatories
    colors = {0: 'orange', 1: 'red', 2: 'green'}
    labels = {0: 'W146', 1: 'Z087', 2: 'K213'}
    
    for obs_code in combined_df['observatory_code'].unique():
        obs_data = combined_df[combined_df['observatory_code'] == obs_code]
        
        plt.subplot(2, 1, 1)
        plt.errorbar(obs_data['BJD'], obs_data['measured_relative_flux'], 
                    yerr=obs_data['measured_relative_flux_error'],
                    fmt='.', color=colors.get(obs_code, 'blue'), 
                    label=labels.get(obs_code, f'Obs {obs_code}'), alpha=0.7)
        
        plt.subplot(2, 1, 2)
        # Plot magnification (assuming baseline flux is minimum)
        baseline = obs_data['measured_relative_flux'].min()
        magnification = obs_data['measured_relative_flux'] / baseline
        plt.errorbar(obs_data['BJD'], magnification,
                    yerr=obs_data['measured_relative_flux_error'] / baseline,
                    fmt='.', color=colors.get(obs_code, 'blue'), alpha=0.7)
    
    plt.subplot(2, 1, 1)
    plt.ylabel('Relative Flux')
    plt.legend()
    plt.title(f'Lightcurve Data - {event_name}' if event_name else 'Lightcurve Data')
    
    plt.subplot(2, 1, 2)
    plt.xlabel('BJD')
    plt.ylabel('Magnification')
    plt.yscale('log')
    
    plt.tight_layout()
    plt.show()
else:
    print("No lightcurve data to plot. Please load data first.")

### Corner Plot WITHOUT Fisher Annotations

This creates a clean corner plot without any Fisher matrix overlays:

In [ ]:
if samples is not None:
    # Parameter labels (adjust based on whether LOM is enabled)
    if samples.shape[1] == 12:  # LOM enabled
        labels = [
            r"$s$", r"$q$", r"$\rho$", 
            r"$u_0$", r"$\alpha$", r"$t_0$", r"$t_E$", 
            r"$\pi_{EE}$", r"$\pi_{EN}$", 
            r"$i$", r"$\phi$", r"$P$"
        ]
    else:  # 9 parameters (no LOM)
        labels = [
            r"$s$", r"$q$", r"$\rho$", 
            r"$u_0$", r"$\alpha$", r"$t_0$", r"$t_E$", 
            r"$\pi_{EE}$", r"$\pi_{EN}$"
        ]
    
    # Get truth values if available
    truth_values = None
    if truths is not None and 'params' in truths:
        truth_values = truths['params'][:samples.shape[1]]
    
    print(f"Creating corner plot for {samples.shape[1]} parameters...")
    print(f"Using {len(samples)} samples")
    
    # Create clean corner plot
    fig = corner.corner(
        samples,
        labels=labels,
        truths=truth_values,
        truth_color='red',
        show_titles=True,
        title_kwargs={"fontsize": 12},
        label_kwargs={"fontsize": 14}
    )
    
    plt.suptitle(f'Posterior Samples - {event_name}' if event_name else 'Posterior Samples', 
                fontsize=16, y=0.98)
    plt.show()
    
    print("Clean corner plot created (no Fisher annotations)")
    
else:
    print("No posterior samples available for corner plot.")
    print("Run the sampler first to generate samples, then load them using the paths at the top.")

### Fisher Matrix Visualization

In [ ]:
if hasattr(data_obj, 'model_covariance') and data_obj.model_covariance is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot covariance matrix
    im1 = ax1.imshow(fisher_cov, cmap='viridis')
    ax1.set_title('Fisher Covariance Matrix')
    ax1.set_xticks(range(len(param_names)))
    ax1.set_yticks(range(len(param_names)))
    ax1.set_xticklabels(param_names, rotation=45)
    ax1.set_yticklabels(param_names)
    plt.colorbar(im1, ax=ax1, shrink=0.8)
    
    # Plot correlation matrix
    im2 = ax2.imshow(fisher_corr, cmap='coolwarm', vmin=-1, vmax=1)
    ax2.set_title('Fisher Correlation Matrix')
    ax2.set_xticks(range(len(param_names)))
    ax2.set_yticks(range(len(param_names)))
    ax2.set_xticklabels(param_names, rotation=45)
    ax2.set_yticklabels(param_names)
    plt.colorbar(im2, ax=ax2, shrink=0.8)
    
    plt.tight_layout()
    plt.show()
else:
    print("No Fisher matrix to visualize.")

## 6. Export Results

Save processed data and analysis results:

In [ ]:
# Save the lightcurve DataFrame if you want
if 'combined_df' in locals() and event_name:
    output_file = f"{event_name}_lightcurve_data.csv"
    combined_df.to_csv(output_file, index=False)
    print(f"Lightcurve data saved to: {output_file}")

# Save Fisher matrix information if available
if hasattr(data_obj, 'model_covariance') and data_obj.model_covariance is not None and event_name:
    fisher_output = {
        'covariance': fisher_cov,
        'uncertainties': fisher_unc,
        'correlation': fisher_corr,
        'parameter_names': param_names
    }
    
    fisher_file = f"{event_name}_fisher_info.pkl"
    with open(fisher_file, 'wb') as f:
        pickle.dump(fisher_output, f)
    print(f"Fisher information saved to: {fisher_file}")

print("\nNotebook execution complete!")
print("You can now experiment with the Fisher stuff in the cells above.")